In [2]:
# ============================================================
# WEEK 11 NOTEBOOK — MODEL EXPLAINABILITY, BIAS, RISK ANALYSIS
# ============================================================


# ============================================================
# Imports
# ============================================================
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier


# ============================================================
# Locate Project Paths
# ============================================================
project_dir = Path.cwd().resolve()
if not (project_dir / "models").exists():
    project_dir = project_dir.parent

model_path = project_dir / "models" / "week9_best_model.pkl"
data_path = project_dir / "data" / "processed" / "matchupDiff_week5_features.csv"

# ============================================================
# Load Final Model Payload (Week 10 Final)
# ============================================================
with open(model_path, "rb") as f:
    week10Model = pickle.load(f)

model_family = week10Model.get("modelFamily", "xgboost")
week10_params_raw = dict(week10Model.get("params", {}))
decision_threshold = float(week10Model["threshold"])
feature_cols = list(week10Model["featureCols"])

print("============================================================")
print("Loaded Week 10 Final Model Payload")
print("============================================================")
print(f"Model Path: {model_path}")
print(f"Model Family: {model_family}")
print(f"Decision Threshold: {decision_threshold}")
print(f"Number of Features: {len(feature_cols)}")


# ============================================================
# Helper: Convert Week 9/10 Param Names -> XGBoost Names
# ============================================================
param_key_map = {
    "nEstimators": "n_estimators",
    "maxDepth": "max_depth",
    "learningRate": "learning_rate",
    "colsampleBytree": "colsample_bytree",
    "minChildWeight": "min_child_weight",
    "regLambda": "reg_lambda",
    "regAlpha": "reg_alpha",
    "randomState": "random_state",
}


# ============================================================
# Build Pipeline (Same Hyperparameters as Week 10 Final)
# ============================================================
def build_week10_pipeline():
    if model_family != "xgboost":
        raise ValueError(f"Week 11 notebook currently supports xgboost only. Found: {model_family}")

    xgb_params = {}
    for k, v in week10_params_raw.items():
        if k in {"modelFamily", "scaleFeatures", "threshold"}:
            continue
        xgb_params[param_key_map.get(k, k)] = v

    # stable defaults
    xgb_params.setdefault("objective", "binary:logistic")
    xgb_params.setdefault("eval_metric", "logloss")
    xgb_params.setdefault("n_jobs", -1)
    xgb_params.setdefault("tree_method", "hist")

    model = XGBClassifier(**xgb_params)

    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model),
    ])


# ============================================================
# Load Dataset
# ============================================================
df = pd.read_csv(data_path)
df["y"] = df["y"].astype(int)

print("\n============================================================")
print("Loaded Dataset")
print("============================================================")
print(f"Dataset Shape: {df.shape}")
display(df.head())


# ============================================================
# Remove Leakage Columns + Define Train/Val/Test Split
# ============================================================
leak_cols = [
    "WScore", "LScore", "WTeamID", "LTeamID", "winnerTeamId",
    "team1Name", "team2Name", "Unnamed: 0"
]
id_cols = ["team1Id", "team2Id"]
drop_cols = ["Season", "y"] + leak_cols + id_cols

train_df = df[df["Season"] <= 2021].copy()
val_df = df[df["Season"] == 2022].copy()
test_df = df[df["Season"].between(2023, 2025)].copy()


def build_xy(split_df, base_feature_cols):
    x = split_df.drop(columns=drop_cols, errors="ignore").select_dtypes(include=[np.number]).copy()
    x = x.reindex(columns=base_feature_cols)
    y = split_df["y"].astype(int).copy()
    return x, y


x_train, y_train = build_xy(train_df, feature_cols)
x_val, y_val = build_xy(val_df, feature_cols)
x_test, y_test = build_xy(test_df, feature_cols)

print("\n============================================================")
print("Train/Validation/Test Split")
print("============================================================")
print(f"Train: {x_train.shape}")
print(f"Validation: {x_val.shape}")
print(f"Test: {x_test.shape}")


# ============================================================
# Fit Model
# ============================================================
pipe = build_week10_pipeline()
pipe.fit(x_train, y_train)

train_proba = pipe.predict_proba(x_train)[:, 1]
val_proba = pipe.predict_proba(x_val)[:, 1]
test_proba = pipe.predict_proba(x_test)[:, 1]


def score_split(y_true, proba, threshold):
    preds = (proba >= threshold).astype(int)
    metrics = {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
    }
    cm = pd.DataFrame(
        confusion_matrix(y_true, preds),
        index=["Actual 0", "Actual 1"],
        columns=["Pred 0", "Pred 1"],
    )
    return metrics, cm


train_metrics, train_cm = score_split(y_train, train_proba, decision_threshold)
val_metrics, val_cm = score_split(y_val, val_proba, decision_threshold)
test_metrics, test_cm = score_split(y_test, test_proba, decision_threshold)

metrics_df = pd.DataFrame([
    {"dataset": "train", **train_metrics},
    {"dataset": "validation", **val_metrics},
    {"dataset": "test", **test_metrics},
])

print("\n============================================================")
print("Model Performance (Week 10 Final Model)")
print("============================================================")
display(metrics_df.round(4))

print("\nTRAIN Confusion Matrix")
display(train_cm)

print("\nVALIDATION Confusion Matrix")
display(val_cm)

print("\nTEST Confusion Matrix")
display(test_cm)


# ============================================================
# PART 1 — FEATURE IMPORTANCE (XGBoost Built-in)
# ============================================================
print("\n============================================================")
print("Feature Importance (XGBoost Built-in)")
print("============================================================")

xgb_model = pipe.named_steps["model"]

importance_df = pd.DataFrame({
    "feature": x_train.columns,
    "importance": xgb_model.feature_importances_
}).sort_values("importance", ascending=False)

print("Top 15 Most Important Features (Built-in):")
display(importance_df.head(15))


# ============================================================
# PART 2 — FEATURE IMPORTANCE (Permutation Importance)
# ============================================================
print("\n============================================================")
print("Permutation Feature Importance (Validation Accuracy Drop)")
print("============================================================")

perm_result = permutation_importance(
    pipe,
    x_val,
    y_val,
    n_repeats=10,
    random_state=42,
    scoring="accuracy"
)

perm_importance_df = pd.DataFrame({
    "feature": x_val.columns,
    "perm_importance_mean": perm_result.importances_mean,
    "perm_importance_std": perm_result.importances_std
}).sort_values("perm_importance_mean", ascending=False)

print("Top 15 Most Important Features (Permutation):")
display(perm_importance_df.head(15))


# ============================================================
# PART 3 — EXTRACT 5 RANDOM INDIVIDUAL PREDICTIONS
# ============================================================
print("\n============================================================")
print("5 Random Individual Predictions (Test Set)")
print("============================================================")

np.random.seed(42)
sample_indices = np.random.choice(x_test.index, size=5, replace=False)

sample_x = x_test.loc[sample_indices]
sample_y = y_test.loc[sample_indices]

sample_proba = pipe.predict_proba(sample_x)[:, 1]
sample_pred = (sample_proba >= decision_threshold).astype(int)

sample_results_df = pd.DataFrame({
    "row_index": sample_indices,
    "true_label": sample_y.values,
    "predicted_label": sample_pred,
    "predicted_probability": sample_proba
})

display(sample_results_df)


# ============================================================
# PART 4 — EXPLAIN EACH PREDICTION USING FEATURE SENSITIVITY
# (Perturbation: slightly increase/decrease top features)
# ============================================================
print("\n============================================================")
print("Individual Prediction Explanation (Feature Sensitivity)")
print("============================================================")


def explain_prediction_by_feature_change(pipe, x_row, baseline_proba, top_features, step=1.0):
    """
    For each feature, increase and decrease it slightly and measure probability impact.
    This gives a report-friendly explanation of which features influence predictions.
    """
    impacts = []

    for feat in top_features:
        modified_plus = x_row.copy()
        modified_minus = x_row.copy()

        modified_plus[feat] += step
        modified_minus[feat] -= step

        plus_proba = pipe.predict_proba(pd.DataFrame([modified_plus]))[:, 1][0]
        minus_proba = pipe.predict_proba(pd.DataFrame([modified_minus]))[:, 1][0]

        impacts.append({
            "feature": feat,
            "baseline_value": x_row[feat],
            "baseline_proba": baseline_proba,
            "proba_if_plus": plus_proba,
            "proba_if_minus": minus_proba,
            "impact_plus": plus_proba - baseline_proba,
            "impact_minus": minus_proba - baseline_proba
        })

    return pd.DataFrame(impacts).sort_values("impact_plus", ascending=False)


top_features_for_explain = perm_importance_df.head(10)["feature"].tolist()

for idx in sample_indices:
    row = sample_x.loc[idx]
    base_prob = pipe.predict_proba(pd.DataFrame([row]))[:, 1][0]
    pred_class = int(base_prob >= decision_threshold)

    print("\n" + "=" * 80)
    print(f"Prediction Explanation for Test Row Index: {idx}")
    print(f"True Label: {sample_y.loc[idx]}")
    print(f"Predicted Probability: {base_prob:.4f}")
    print(f"Predicted Class (threshold={decision_threshold}): {pred_class}")

    explain_df = explain_prediction_by_feature_change(
        pipe,
        row,
        base_prob,
        top_features=top_features_for_explain,
        step=1.0
    )

    print("\nTop Feature Sensitivities (higher impact means more influential):")
    display(explain_df.head(10))


# ============================================================
# PART 5 — FLIP ANALYSIS
# Find which feature change would flip the prediction
# ============================================================
print("\n============================================================")
print("Flip Analysis (How Much Must Change to Flip Prediction?)")
print("============================================================")


def find_flip_feature(pipe, x_row, threshold, feature_list, max_change=50, step=1.0):
    """
    Incrementally adjusts one feature until prediction flips.
    Returns the smallest change needed for each feature.
    """
    base_prob = pipe.predict_proba(pd.DataFrame([x_row]))[:, 1][0]
    base_pred = int(base_prob >= threshold)

    results = []

    for feat in feature_list:
        for direction in [+1, -1]:
            temp = x_row.copy()

            for k in range(1, int(max_change / step) + 1):
                temp[feat] = x_row[feat] + direction * k * step
                prob = pipe.predict_proba(pd.DataFrame([temp]))[:, 1][0]
                pred = int(prob >= threshold)

                if pred != base_pred:
                    results.append({
                        "feature": feat,
                        "direction": "increase" if direction == 1 else "decrease",
                        "change_needed": direction * k * step,
                        "new_probability": prob
                    })
                    break

    df_results = pd.DataFrame(results)

    if df_results.empty:
        return df_results

    return df_results.sort_values("change_needed", key=lambda s: s.abs())


flip_features = perm_importance_df.head(8)["feature"].tolist()

for idx in sample_indices:
    row = sample_x.loc[idx]
    base_prob = pipe.predict_proba(pd.DataFrame([row]))[:, 1][0]
    base_pred = int(base_prob >= decision_threshold)

    print("\n" + "=" * 80)
    print(f"Flip Analysis for Test Row Index: {idx}")
    print(f"Baseline Probability: {base_prob:.4f}")
    print(f"Baseline Predicted Class: {base_pred}")

    flip_df = find_flip_feature(
        pipe,
        row,
        decision_threshold,
        flip_features,
        max_change=200,
        step=1.0
    )

    if flip_df.empty:
        print("No flip found within the allowed change range.")
    else:
        print("Smallest Feature Adjustments That Flip Prediction (Approximate):")
        display(flip_df.head(10))


# ============================================================
# PART 6 — BIAS ANALYSIS (PROXY GROUPING)
# Instead we evaluate bias by matchup difficulty groups.
# Proxy Group: abs(seed difference)
# ============================================================
print("\n============================================================")
print("Bias Analysis (Seed Difference Group Proxy)")
print("============================================================")

bias_df = test_df.copy()

if "Seed_diff" in bias_df.columns:
    bias_df["abs_seed_diff"] = bias_df["Seed_diff"].abs()

    bias_df["seed_group"] = pd.cut(
        bias_df["abs_seed_diff"],
        bins=[-1, 2, 5, 10, 20, 100],
        labels=["Very Close (0-2)", "Close (3-5)", "Moderate (6-10)", "Large (11-20)", "Extreme (21+)"]
    )
else:
    bias_df["seed_group"] = "Unknown"

bias_x, bias_y = build_xy(bias_df, feature_cols)

bias_proba = pipe.predict_proba(bias_x)[:, 1]
bias_pred = (bias_proba >= decision_threshold).astype(int)

bias_df["pred"] = bias_pred
bias_df["proba"] = bias_proba
bias_df["actual"] = bias_y.values


def group_bias_metrics(df, group_col):
    out = []

    for g, sub in df.groupby(group_col):
        if len(sub) < 5:
            continue

        acc = accuracy_score(sub["actual"], sub["pred"])
        prec = precision_score(sub["actual"], sub["pred"], zero_division=0)
        rec = recall_score(sub["actual"], sub["pred"], zero_division=0)

        pos_rate = sub["pred"].mean()
        actual_rate = sub["actual"].mean()

        out.append({
            "group": str(g),
            "n": len(sub),
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "predicted_positive_rate": pos_rate,
            "actual_positive_rate": actual_rate,
            "difference_pred_minus_actual": pos_rate - actual_rate
        })

    return pd.DataFrame(out).sort_values("n", ascending=False)


bias_metrics = group_bias_metrics(bias_df, "seed_group")

print("Bias Metrics by Seed Group:")
display(bias_metrics.round(4))


# ============================================================
# Bias Summary Statistic: Disparate Impact Style Ratio
# ============================================================
if not bias_metrics.empty and len(bias_metrics) > 1:
    max_pos = bias_metrics["predicted_positive_rate"].max()
    min_pos = bias_metrics["predicted_positive_rate"].min()

    print("\n============================================================")
    print("Bias Summary (Predicted Positive Rate Spread)")
    print("============================================================")
    print(f"Max predicted positive rate: {max_pos:.4f}")
    print(f"Min predicted positive rate: {min_pos:.4f}")
    print(f"Difference: {max_pos - min_pos:.4f}")

    if max_pos > 0:
        print(f"Disparate Impact Ratio (min/max): {min_pos / max_pos:.4f}")


# ============================================================
# OPTIONAL EXTRA CREDIT — BIAS MITIGATION STRATEGY
# Strategy: upweight close matchups in training
# ============================================================
print("\n============================================================")
print("OPTIONAL EXTRA CREDIT — Bias Mitigation Retraining")
print("============================================================")

train_bias_df = train_df.copy()

if "Seed_diff" in train_bias_df.columns:
    train_bias_df["abs_seed_diff"] = train_bias_df["Seed_diff"].abs()

    # upweight close games
    train_bias_df["weight"] = np.where(train_bias_df["abs_seed_diff"] <= 2, 2.0, 1.0)
else:
    train_bias_df["weight"] = 1.0

x_train_bias, y_train_bias = build_xy(train_bias_df, feature_cols)
sample_weights = train_bias_df["weight"].values

pipe_bias = build_week10_pipeline()

pipe_bias.fit(
    x_train_bias,
    y_train_bias,
    model__sample_weight=sample_weights
)

train_proba_b = pipe_bias.predict_proba(x_train)[:, 1]
val_proba_b = pipe_bias.predict_proba(x_val)[:, 1]
test_proba_b = pipe_bias.predict_proba(x_test)[:, 1]

train_metrics_b, _ = score_split(y_train, train_proba_b, decision_threshold)
val_metrics_b, _ = score_split(y_val, val_proba_b, decision_threshold)
test_metrics_b, _ = score_split(y_test, test_proba_b, decision_threshold)

metrics_bias_model_df = pd.DataFrame([
    {"dataset": "train", **train_metrics_b},
    {"dataset": "validation", **val_metrics_b},
    {"dataset": "test", **test_metrics_b},
])

print("Performance Metrics After Bias-Mitigation Retraining:")
display(metrics_bias_model_df.round(4))


# ============================================================
# Compare Week 10 Final vs Bias-Mitigation Model
# ============================================================
compare_df = metrics_df.copy()
compare_df["model"] = "Week10 Final"

bias_compare_df = metrics_bias_model_df.copy()
bias_compare_df["model"] = "Bias Mitigation Retrain"

combined_compare = pd.concat([compare_df, bias_compare_df], ignore_index=True)
combined_compare = combined_compare[["model", "dataset", "accuracy", "precision", "recall"]]

print("\n============================================================")
print("Week10 Final vs Bias-Mitigation Retrain Comparison")
print("============================================================")
display(combined_compare.round(4))


# ============================================================
# Re-run Bias Metrics on Test After Mitigation
# ============================================================
bias_proba2 = pipe_bias.predict_proba(bias_x)[:, 1]
bias_pred2 = (bias_proba2 >= decision_threshold).astype(int)

bias_df2 = bias_df.copy()
bias_df2["pred"] = bias_pred2
bias_df2["proba"] = bias_proba2

bias_metrics_after = group_bias_metrics(bias_df2, "seed_group")

print("\n============================================================")
print("Bias Metrics After Mitigation (Test Set)")
print("============================================================")
display(bias_metrics_after.round(4))


# ============================================================
# Save Outputs
# ============================================================
output_dir = project_dir / "models"
output_dir.mkdir(exist_ok=True)

importance_df.to_csv(output_dir / "week11_xgb_feature_importance.csv", index=False)
perm_importance_df.to_csv(output_dir / "week11_permutation_importance.csv", index=False)
bias_metrics.to_csv(output_dir / "week11_bias_metrics_before.csv", index=False)
bias_metrics_after.to_csv(output_dir / "week11_bias_metrics_after.csv", index=False)
combined_compare.to_csv(output_dir / "week11_model_comparison_week10_vs_bias_mitigated.csv", index=False)
sample_results_df.to_csv(output_dir / "week11_sample_predictions.csv", index=False)

print("\n============================================================")
print("Saved Week 11 Outputs")
print("============================================================")
print(f"Saved to folder: {output_dir}")
print("Files saved:")
print("- week11_xgb_feature_importance.csv")
print("- week11_permutation_importance.csv")
print("- week11_bias_metrics_before.csv")
print("- week11_bias_metrics_after.csv")
print("- week11_model_comparison_week10_vs_bias_mitigated.csv")
print("- week11_sample_predictions.csv")

Loaded Week 10 Final Model Payload
Model Path: /Users/ashererickson/models/week9_best_model.pkl
Model Family: xgboost
Decision Threshold: 0.45
Number of Features: 108

Loaded Dataset
Dataset Shape: (1353, 120)


,Unnamed: 0,Season,DayNum,WTeamID,LTeamID,WScore,LScore,team1Id,team2Id,y,...,oppFgPct,oppThreePtPct,oppFtPct,oppEFgPct,oppAstTo,fgPctDiff,threePtPctDiff,ftPctDiff,eFgPctDiff,astToDiff
0,0,2003,134,1421,1411,92,84,1411,1421,0,...,42.0,32.0,71.1,48.1,1.0,1.3,6.7,-25.9,4.1,0.1
1,1,2003,136,1112,1436,80,51,1112,1436,1,...,31.2,25.0,100.0,34.4,0.7,14.6,12.3,-22.1,17.1,0.7
2,2,2003,136,1113,1272,84,71,1113,1272,0,...,36.2,25.0,66.7,41.3,0.9,9.9,9.5,4.4,8.7,0.5
3,3,2003,136,1141,1166,79,73,1141,1166,0,...,45.0,41.2,70.6,50.8,1.0,0.5,4.6,1.2,-0.3,-0.4
4,4,2003,136,1143,1301,76,74,1143,1301,1,...,44.6,42.9,75.0,52.7,1.1,2.0,-9.6,-15.0,-1.0,-0.1



Train/Validation/Test Split
Train: (1162, 108)
Validation: (63, 108)
Test: (128, 108)

Model Performance (Week 10 Final Model)


,dataset,accuracy,precision,recall
0,train,0.9165,0.8944,0.9936
1,validation,0.8095,0.7800,0.9750
2,test,0.7031,0.7156,0.9176



TRAIN Confusion Matrix


...

,model,dataset,accuracy,precision,recall
0,Week10 Final,train,0.9165,0.8944,0.9936
1,Week10 Final,validation,0.8095,0.7800,0.9750
2,Week10 Final,test,0.7031,0.7156,0.9176
3,Bias Mitigation Retrain,train,0.9165,0.8944,0.9936
4,Bias Mitigation Retrain,validation,0.7778,0.7708,0.9250
5,Bias Mitigation Retrain,test,0.7109,0.7143,0.9412



Bias Metrics After Mitigation (Test Set)


/var/folders/6s/9hp8rr2j1b3ck1zx3w7rc6mr0000gn/T/ipykernel_1627/1723530406.py:439: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for g, sub in df.groupby(group_col):


,group,n,accuracy,precision,recall,predicted_positive_rate,actual_positive_rate,difference_pred_minus_actual
2,Moderate (6-10),38,0.7895,0.7838,1.0000,0.9737,0.7632,0.2105
1,Close (3-5),37,0.5946,0.6333,0.8261,0.8108,0.6216,0.1892
0,Very Close (0-2),29,0.6207,0.5238,0.9167,0.7241,0.4138,0.3103
3,Large (11-20),24,0.8750,0.8750,1.0000,1.0000,0.8750,0.1250



Saved Week 11 Outputs
Saved to folder: /Users/ashererickson/Downloads/models
Files saved:
- week11_xgb_feature_importance.csv
- week11_permutation_importance.csv
- week11_bias_metrics_before.csv
- week11_bias_metrics_after.csv
- week11_model_comparison_week10_vs_bias_mitigated.csv
- week11_sample_predictions.csv
